In [9]:
# ============================================================
# Gold Layer: Star Schema — dim_regions
# Using a Window function for stable, sequential surrogate keys
# ============================================================

from pyspark.sql.functions import (
    col, current_timestamp, row_number
)
from pyspark.sql.window import Window

# Check your table path/namespace (adjust if you aren't using a catalog/schema named Silver_Lakehouse)
df_silver = spark.table("Silver_lakehouse.dbo.silver_countries")

# ---- dim_regions ----
# 1. Get distinct region and subregion combinations, filtering out "N/A"
df_dim_regions = (
    df_silver.select("region", "subregion")
    .distinct()
    .filter(col("region") != "N/A")
)

# 2. Define a deterministic window ordering to generate stable surrogate keys
window_spec = Window.orderBy("region", "subregion")

# 3. Assign a deterministic, sequential surrogate key (1, 2, 3...)
df_dim_regions = df_dim_regions.withColumn(
    "region_key", row_number().over(window_spec)
).select(
    col("region_key"),
    col("region"),
    col("subregion"),
    current_timestamp().alias("dwh_created_date")
)

# Write to Gold Delta table
df_dim_regions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_regions")

print(f"dim_regions: {df_dim_regions.count()} rows")
df_dim_regions.show(20, truncate=False)

StatementMeta(, 250d4049-41ea-40e6-b4bd-60d593b283ba, 11, Finished, Available, Finished, False)

dim_regions: 25 rows
+----------+---------+------------------+--------------------------+
|region_key|region   |subregion         |dwh_created_date          |
+----------+---------+------------------+--------------------------+
|1         |Africa   |Eastern Africa    |2026-10-25 16:38:29.766402|
|2         |Africa   |Middle Africa     |2026-10-25 16:38:29.766402|
|3         |Africa   |Northern Africa   |2026-10-25 16:38:29.766402|
|4         |Africa   |Southern Africa   |2026-10-25 16:38:29.766402|
|5         |Africa   |Western Africa    |2026-10-25 16:38:29.766402|
|6         |Americas |Caribbean         |2026-10-25 16:38:29.766402|
|7         |Americas |Central America   |2026-10-25 16:38:29.766402|
|8         |Americas |North America     |2026-10-25 16:38:29.766402|
|9         |Americas |South America     |2026-10-25 16:38:29.766402|
|10        |Antarctic|                  |2026-10-25 16:38:29.766402|
|11        |Asia     |Central Asia      |2026-10-25 16:38:29.766402|
|12        |A

In [11]:
# ============================================================
# Gold Layer: Star Schema — dim_countries
# Master country dimension with region foreign key
# ============================================================

from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.window import Window

# 2. Define a deterministic window ordering to generate stable surrogate keys
window_spec = Window.orderBy("region", "subregion")

# 3. Assign a deterministic, sequential surrogate key (1, 2, 3...)
df_dim_regions = df_dim_regions.withColumn(
    "region_key", row_number().over(window_spec)
).select(
    col("region_key"),
    col("region"),
    col("subregion"),
    current_timestamp().alias("dwh_created_date")
)



# Read from Silver layer (adjust catalog/schema namespace if needed)
df_silver = spark.table("Silver_Lakehouse.dbo.silver_countries")

# Join with dim_regions to get the region surrogate key
df_dim_countries = df_silver.join(
    df_dim_regions.select("region_key", "region", "subregion"),
    on=["region", "subregion"],
    how="left"
).select(
    col("country_code_3").alias("country_key"),    # Natural PK
    col("country_code_2"),
    col("country_code_numeric"),
    col("country_name"),
    col("official_name"),
    col("capital"),
    col("region"),
    col("subregion"),
    col("continent"),
    col("region_key"),                             # Foreign Key to dim_regions
    col("latitude"),
    col("longitude"),
    col("is_landlocked"),
    col("is_un_member"),
    col("primary_timezone"),
    col("primary_currency_code"),
    col("language_count"),
    col("border_count"),
    col("flag_emoji"),
    col("google_maps_url"),
    current_timestamp().alias("dwh_created_date")
)

# Write to Gold Delta table
df_dim_countries.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_countries")

print(f"dim_countries: {df_dim_countries.count()} rows")
df_dim_countries.show(5, truncate=40)

StatementMeta(, 250d4049-41ea-40e6-b4bd-60d593b283ba, 13, Finished, Available, Finished, False)

dim_countries: 254 rows
+-----------+--------------+--------------------+-------------------+----------------------------------+-------------+--------+------------------+-------------+----------+-----------+------------+-------------+------------+----------------+---------------------+--------------+------------+----------+-------------------------------------+--------------------------+
|country_key|country_code_2|country_code_numeric|       country_name|                     official_name|      capital|  region|         subregion|    continent|region_key|   latitude|   longitude|is_landlocked|is_un_member|primary_timezone|primary_currency_code|language_count|border_count|flag_emoji|                      google_maps_url|          dwh_created_date|
+-----------+--------------+--------------------+-------------------+----------------------------------+-------------+--------+------------------+-------------+----------+-----------+------------+-------------+------------+----------------+--

In [14]:
# ============================================================
# Gold Layer: Star Schema — fact_country_metrics
# Main fact table containing numeric indicators and categories
# ============================================================

from pyspark.sql.functions import (
    col, when, lit, current_timestamp, 
    round as spark_round
)

# Read from silver layer
df_silver = spark.table("Silver_Lakehouse.dbo.silver_countries")

df_fact_metrics = df_silver.select(
    col("country_code_3").alias("country_key"),
    col("population"),
    col("area_km2"),
    col("border_count"),
    col("language_count"),

    # Calculated: population density (people per km2) with zero-division safety
    when(
        (col("area_km2").isNotNull()) & (col("area_km2") > 0),
        spark_round(col("population") / col("area_km2"), 2)
    ).otherwise(lit(None)).alias("population_density"),

    # Size classification based on area
    when(col("area_km2") >= 1000000, "Very Large")
    .when(col("area_km2") >= 100000,  "Large")
    .when(col("area_km2") >= 10000,   "Medium")
    .when(col("area_km2") >= 1000,    "Small")
    .otherwise("Very Small").alias("size_category"),

    # Population classification
    when(col("population") >= 100000000, "100M+")
    .when(col("population") >= 50000000,  "50M-100M")
    .when(col("population") >= 10000000,  "10M-50M")
    .when(col("population") >= 1000000,   "1M-10M")
    .when(col("population") >= 100000,    "100K-1M")
    .otherwise("Under 100K").alias("population_category"),

    current_timestamp().alias("dwh_created_date")
)

# Write to Gold Delta table
df_fact_metrics.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_country_metrics")

print(f"fact_country_metrics: {df_fact_metrics.count()} rows")
df_fact_metrics.show(10)

StatementMeta(, 250d4049-41ea-40e6-b4bd-60d593b283ba, 41, Finished, Available, Finished, False)

fact_country_metrics: 254 rows
+-----------+----------+--------+------------+--------------+------------------+-------------+-------------------+--------------------+
|country_key|population|area_km2|border_count|language_count|population_density|size_category|population_category|    dwh_created_date|
+-----------+----------+--------+------------+--------------+------------------+-------------+-------------------+--------------------+
|        TLS|   1409452| 14874.0|           1|             2|             94.76|       Medium|             1M-10M|2026-10-25 16:47:...|
|        TGO|   8095498| 56785.0|           3|             1|            142.56|       Medium|             1M-10M|2026-10-25 16:47:...|
|        TKL|      1411|    12.0|           0|             3|            117.58|   Very Small|         Under 100K|2026-10-25 16:47:...|
|        TON|    100179|   747.0|           0|             2|            134.11|   Very Small|            100K-1M|2026-10-25 16:47:...|
|        TTO|   1

In [15]:
# ---- fact_country_languages ----
df_silver_lang = spark.table("Silver_Lakehouse.dbo.silver_country_languages")

df_fact_languages = df_silver_lang.select(
    col("country_code_3")   .alias("country_key"),
    col("language_code"),
    col("language_name"),
    current_timestamp()     .alias("dwh_created_date")
)

df_fact_languages.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_country_languages")

print(f"fact_country_languages: {df_fact_languages.count()} rows")

# ---- fact_country_currencies ----
df_silver_curr = spark.table("Silver_Lakehouse.dbo.silver_country_currencies")

df_fact_currencies = df_silver_curr.select(
    col("country_code_3")   .alias("country_key"),
    col("currency_code"),
    col("currency_name"),
    col("currency_symbol"),
    current_timestamp()     .alias("dwh_created_date")
)

df_fact_currencies.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_country_currencies")

print(f"fact_country_currencies: {df_fact_currencies.count()} rows")

StatementMeta(, 250d4049-41ea-40e6-b4bd-60d593b283ba, 43, Finished, Available, Finished, False)

fact_country_languages: 427 rows
fact_country_currencies: 276 rows


In [17]:
print("=== GOLD LAYER VALIDATION ===\n")

gold_tables = [
    "dim_regions", "dim_countries",
    "fact_country_metrics", "fact_country_languages",
    "fact_country_currencies"
]

for t in gold_tables:
    count = spark.table(t).count()
    print(f"{t}: {count} rows")

# Star schema join test
print("\n=== STAR SCHEMA JOIN TEST ===")
result = spark.table("fact_country_metrics").alias("f") \
    .join(spark.table("dim_countries").alias("d"),
          col("f.country_key") == col("d.country_key")) \
    .join(spark.table("dim_regions").alias("r"),
          col("d.region_key") == col("r.region_key")) \
    .select(
        col("d.country_name"),
        col("d.region"),
        col("d.subregion"),
        col("f.population"),
        col("f.area_km2"),
        col("f.population_density"),
        col("f.size_category")
    ).orderBy("f.population", ascending=False)

result.show(15, truncate=30)

print("\n=== BUSINESS METRICS ===")
metrics = spark.table("fact_country_metrics")
print(f"Total world population: {metrics.agg({'population': 'sum'}).collect()[0][0]:,}")
print(f"Total countries: {metrics.count()}")
print(f"Average area km2: {metrics.agg({'area_km2': 'avg'}).collect()[0][0]:,.0f}")

StatementMeta(, 250d4049-41ea-40e6-b4bd-60d593b283ba, 45, Finished, Available, Finished, False)

=== GOLD LAYER VALIDATION ===

dim_regions: 25 rows
dim_countries: 254 rows
fact_country_metrics: 254 rows
fact_country_languages: 427 rows
fact_country_currencies: 276 rows

=== STAR SCHEMA JOIN TEST ===
+-------------+--------+------------------+----------+-----------+------------------+-------------+
| country_name|  region|         subregion|population|   area_km2|population_density|size_category|
+-------------+--------+------------------+----------+-----------+------------------+-------------+
|        India|    Asia|     Southern Asia|1429404000|  3287590.0|            434.79|   Very Large|
|        China|    Asia|      Eastern Asia|1404890000|  9706961.0|            144.73|   Very Large|
|United States|Americas|     North America| 341784857|  9372610.0|             36.47|   Very Large|
|    Indonesia|    Asia|South-Eastern Asia| 288315089|  1904569.0|            151.38|   Very Large|
|     Pakistan|    Asia|     Southern Asia| 241499431|   881912.0|            273.84|        La